# Trascrivi Lezioni — Google Colab

Turn your lectures (audio/video **files** or **links** like YouTube/Webex) into
cleaned, translated **Word documents** — entirely in the cloud. **Nothing to install**,
and no programming needed. Powered by Google Gemini.

### How this page works (read once)
This is a **Colab notebook**: a page made of **cells**. The grey cells are code; you run
one by clicking the **▶ (play)** button on its left (or pressing **Shift+Enter**).
**Just run them in order, top to bottom.** Each step has a short explanation right above
its cell. Wait for a cell to finish (a green **✓** appears) before running the next one.
The first time, Google may ask you to **authorize / "Run anyway"** — that's normal.

*You never read or edit any code, and you never type file names.*

> **Privacy:** your audio and transcripts are sent to Google Gemini for processing. Only
> transcribe recordings you are allowed to use. A free CPU runtime is enough.


## Step 1 — Install (run once)
Click **▶** on the cell below and wait about a minute. It sets everything up for this session.


In [ ]:
#@title ▶ Install { display-mode: "form" }
!pip install -q "git+https://github.com/GiorgioAlo/lecture-transcriber"
print("Installed. Move on to Step 2.")


## Step 2 — Your Google Gemini API key
The API key is a **free** personal "password" that lets the app use Google Gemini.

**A) Get the key**
1. Open **https://aistudio.google.com/apikey** and **sign in** with your Google account.
2. Click **Create API key** (top right).
3. Give it a **name** (the suggested *Gemini API Key* is fine).
4. Choose a **project** (or click *Create project*, give it a name, create it).
5. The key appears — **copy it** (a long string like `AIza...`). Keep it private.

**B) Give it to the notebook — recommended: store it hidden as a Colab *Secret***
1. On the **left** of this page, click the **🔑 key icon** ("Secrets").
2. **＋ Add new secret** → **Name:** `GOOGLE_API_KEY` → **Value:** paste your key.
3. Turn **ON** the *Notebook access* switch.

Then run the cell below. *(If you skip the Secret, the cell simply asks you to paste the key.)*


In [ ]:
#@title ▶ Load your API key { display-mode: "form" }
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key (hidden): ")
    print("API key set for this session.")


## Step 3 — Add your lectures
**Best way (fast and reliable for big videos):** click the **📁 Files** icon on the **left**
of this page, then **drag your lecture files** into it and wait for the progress bar. For
**Webex**, drop a `cookies.txt` there too. The notebook finds these files automatically.

**Links** (YouTube/Webex): nothing to upload — just paste the link in Step 4.

The cell below is an **optional** upload button (for big files the Files panel on the left is smoother).


In [ ]:
#@title ▶ (optional) Upload via button { display-mode: "form" }
from google.colab import files
uploaded = files.upload()
if uploaded:
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Nothing uploaded here — that's fine if you used the Files panel on the left.")


## Step 4 — Options
Run the cell below and fill the small form on its right:
- **LINKS** — paste any YouTube/Webex links, separated by spaces (leave empty if you only use files).
- **LANGUAGES** — the languages of the Word documents, e.g. `it en`.

The subject of each lecture is detected automatically.


In [ ]:
#@title ▶ Options { display-mode: "form" }
#@markdown **LINKS** (YouTube/Webex), separated by spaces — leave empty if you only use uploaded files:
LINKS = "" #@param {type:"string"}
#@markdown **LANGUAGES** of the output documents (space-separated, e.g. `it en` or `it en fr`):
LANGUAGES = "it en" #@param {type:"string"}
print("Links:", LINKS or "(none)", "| Languages:", LANGUAGES)


## Step 5 — Transcribe
Click **▶** and wait. The notebook processes every lecture you added (uploaded files + links),
one after another — it can take a few minutes each (that is Google doing the work). When it
finishes, your Word documents are offered to you as a **`.zip`** to download.

Once the `.zip` has downloaded, the uploaded videos that were fully processed are **removed
from `/content`** to keep things tidy (anything that failed is kept, so you can retry it).


In [ ]:
#@title ▶ Transcribe (then download the .zip) { display-mode: "form" }
import os, shutil
from pathlib import Path
from trascrittore import Job, esegui, config
from trascrittore.progress import ReporterConsole

key = os.environ.get("GOOGLE_API_KEY", "")
assert key, "Run Step 2 (API key) first."

OUTPUT_DIR = "/content/transcriptions"
WORK = "/content/.cache"
MEDIA = (".mp4", ".m4a", ".mp3", ".wav", ".aac", ".ogg", ".flac", ".mkv", ".mov", ".webm")

content = Path("/content")
file_lezioni = sorted(p for p in content.iterdir() if p.is_file() and p.suffix.lower() in MEDIA)
links = [u for u in LINKS.replace(",", " ").split() if u.lower().startswith(("http://", "https://"))]
cookies = "/content/cookies.txt" if Path("/content/cookies.txt").exists() else None
langs = LANGUAGES.split() or ["it", "en"]

sources = [("file", str(p)) for p in file_lezioni] + [("link", u) for u in links]
if not sources:
    raise SystemExit("No lectures found. Upload files (Files panel on the left) or add links in Step 4.")

print("Lectures to process:")
for kind, s in sources:
    print("  -", Path(s).name if kind == "file" else s)

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
saved, failed = [], 0
ok_files = []   # uploaded files fully processed -> removed from /content AFTER download
for i, (kind, src) in enumerate(sources, 1):
    name = Path(src).stem if kind == "file" else "Lecture"
    print()
    print(f"=== [{i}/{len(sources)}] {name} ===")
    job = Job(
        sorgente=Path(src) if kind == "file" else None,
        url=src if kind == "link" else None,
        nome=name, lingue_output=langs, cookies=cookies,
    )
    try:
        ris = esegui(job, key, ReporterConsole(config.FASI), Path(WORK))
    except Exception as e:
        print("  [x] failed:", e); failed += 1; continue
    for d in ris.documenti:
        dest = out / Path(d).name
        shutil.copy2(d, dest); saved.append(dest)
        print("  [doc]", dest)
    if kind == "file" and ris.completo:
        ok_files.append(Path(src))   # fully done -> safe to remove after download

print()
msg = f"Done. {len(saved)} document(s) saved."
if failed:
    msg += f" ({failed} failed)"
print(msg)

if saved:
    z = shutil.make_archive("/content/transcriptions", "zip", OUTPUT_DIR)
    from google.colab import files as _dl
    _dl.download(z)   # browser download of the .zip
    # Only AFTER the download: remove the uploaded files that were fully processed,
    # to keep /content tidy. Partial/failed lectures are left in place to retry.
    removed = 0
    for f in ok_files:
        try:
            f.unlink(); removed += 1
        except OSError:
            pass
    if removed:
        print(f"Cleaned up: removed {removed} processed file(s) from /content.")
